# Method 2: Decision Trees

Welcome to this notebook focused on classifying sign language gestures using a Decision Tree Classifier. Our goal is to train a model that can recognize hand signs from images and correctly predict the corresponding letter or symbol.

Decision trees offer a simple yet effective approach to classification, and while they may not match the performance of deep learning models for image data, they provide a clear and interpretable baseline for understanding the structure of the problem.

## 1. Setup


### 1.1 Import Libraries


In [2]:
import os
import csv
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_graphviz

### 1.2 Loading the Dataset

Firstly we need to load the datasset using a pytorch container class. This code was already provided to us.

In [3]:
# Setting the path of the training dataset (that was already provided to you)

running_local = True if os.getenv('JUPYTERHUB_USER') is None else False
DATASET_PATH = "../data/sign_lang_train"

# Set the location of the dataset
if running_local:
    # If running on your local machine, the sign_lang_train folder's path should be specified here
    local_path = "sign_lang_train"
    if os.path.exists(local_path):
        DATASET_PATH = local_path
else:
    # If running on the Jupyter hub, this data folder is already available
    # You DO NOT need to upload the data!
    DATASET_PATH = "/data/mlproject22/sign_lang_train"

In [4]:
# Utility function

def read_csv(csv_file):
    with open(csv_file, newline='') as f:
        reader = csv.reader(f)
        data = list(reader)
    return data

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, utils, io
from torchvision.utils import make_grid

from string import ascii_lowercase

class SignLangDataset(Dataset):
    """Sign language dataset"""

    def __init__(self, csv_file, root_dir, class_index_map=None, transform=None):
        """
        Args:
            csv_file (string): Path to the csv file with annotations.
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.data = read_csv(os.path.join(root_dir,csv_file))
        self.root_dir = root_dir
        self.class_index_map = class_index_map
        self.transform = transform
        # List of class names in order
        self.class_names = list(map(str, list(range(10)))) + list(ascii_lowercase)

    def __len__(self):
        """
        Calculates the length of the dataset-
        """
        return len(self.data)

    def __getitem__(self, idx):
        """
        Returns one sample (dict consisting of an image and its label)
        """
        if torch.is_tensor(idx):
            idx = idx.tolist()

        # Read the image and labels
        image_path = os.path.join(self.root_dir, self.data[idx][1])
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        # Shape of the image should be H,W,C where C=1
        image = np.expand_dims(image, 0)
        # The label is the index of the class name in the list ['0','1',...,'9','a','b',...'z']
        # because we should have integer labels in the range 0-35 (for 36 classes)
        label = self.class_names.index(self.data[idx][0])
                
        sample = {'image': image, 'label': label}

        #if self.transform:
        #    sample = self.transform(sample)

        return sample

In [6]:
from torch.utils.data import random_split, DataLoader

dataset = SignLangDataset(csv_file='labels.csv', root_dir=DATASET_PATH)

### 2.4 Augment the Data 

Augmenting the data in such a way, that all labels are represented equally proved to increase the accuracy of all estimations significantly. 
For a detailed explaination of this code, we refer to the cnn_notebook in this project which explains it step by step. Here i just augment the data to have increased variance and equal representation of all classes.

In [7]:
from collections import defaultdict

clustered_data = defaultdict(list)

max_bucket_size = 0

for data in dataset:
    clustered_data[data['label']].append(data)

for bucket in clustered_data.values():
    len_bucket = len(bucket)
    if len_bucket > max_bucket_size:
        max_bucket_size = len_bucket

for label, bucket in clustered_data.items():
    if len(bucket) < max_bucket_size - max_bucket_size * 0.1:
        print(f"To few Images of label: {label}; counted: {len(bucket)}")

To few Images of label: 7; counted: 112
To few Images of label: 18; counted: 280
To few Images of label: 31; counted: 280
To few Images of label: 17; counted: 112
To few Images of label: 13; counted: 168
To few Images of label: 2; counted: 112
To few Images of label: 29; counted: 104
To few Images of label: 33; counted: 112
To few Images of label: 23; counted: 112
To few Images of label: 14; counted: 112
To few Images of label: 26; counted: 112
To few Images of label: 5; counted: 112
To few Images of label: 24; counted: 112
To few Images of label: 28; counted: 280
To few Images of label: 8; counted: 168
To few Images of label: 20; counted: 168
To few Images of label: 22; counted: 112
To few Images of label: 19; counted: 280
To few Images of label: 11; counted: 280
To few Images of label: 34; counted: 280
To few Images of label: 27; counted: 112
To few Images of label: 15; counted: 112
To few Images of label: 32; counted: 112
To few Images of label: 10; counted: 112
To few Images of lab

In [8]:
from torchvision import transforms
from torchvision.transforms import v2
from PIL import Image

augmented_data = []


augment = transforms.Compose([
    v2.RandomPerspective(distortion_scale=0.15),
    v2.RandomRotation(degrees=27),
    v2.RandomApply([v2.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))]),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # or match original data distribution4
])

max_show = 8

for label, bucket in clustered_data.items():
    len_bucket = len(bucket)

    if len_bucket < max_bucket_size - max_bucket_size * 0.1:
        needed = int((max_bucket_size - len_bucket) * random.uniform(0.9, 1.1))

        for i in range(needed):
            img_np = random.choice(bucket)['image']
            img_2d = np.squeeze(img_np, axis=0)
            img_pil = Image.fromarray(img_2d.astype(np.uint8), mode='L')

            augmented_img = augment(img_pil)
            augmented_np = augmented_img.numpy()

            augmented_data.append({'image': augmented_np, 'label': label})

print(f"Augmented {len(augmented_data)} images.")

Augmented 10525 images.


In [9]:
class CombinedDataset(Dataset):
    def __init__(self, original_dataset, augmentation):
        self.original_dataset = original_dataset
        self.augmented_data = augmentation
        self.class_names = original_dataset.class_names
    def __len__(self):
        return len(self.original_dataset) + len(self.augmented_data)

    def __getitem__(self, idx):
        if idx < len(self.original_dataset):
            return self.original_dataset[idx]
        else:
            aug_idx = idx - len(self.original_dataset)
            return self.augmented_data[aug_idx]

In [10]:
augmented_dataset = CombinedDataset(dataset, augmented_data)


### 1.3 Splitting the Data

Now that the dataset is properly set up, it is time to split the data.
Because a final test set is already implemented on the seminars jupyterhub, the data is only split into test and validation sets.

Split Ratio: 80-20

In [11]:
N = len(augmented_dataset) 

torch.manual_seed(92)

train_size = int(0.8*N)
validation_size = N - train_size 

train_dataset, val_dataset = random_split(augmented_dataset, [train_size,  validation_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

## 3. Training the model

In this section i will review different approaches regarding this classification Problems with respect to a solution using decision trees.

### 3.1 Using regular Decision Trees

Let's train a normal decision tree on the dataset using scikit. This will help us get an innitaial value for accuradcy on which we can improve upon.

In [12]:
# Convert PyTorch datasets to NumPy arrays
def dataset_to_numpy(dataset):
    images = []
    labels = []
    for item in dataset:
        image = item['image']
        label = item['label']

        # Convert image to NumPy if it's not already
        if isinstance(image, torch.Tensor):
            image = image.numpy()

        # Flatten image: (1, H, W) -> (H*W,)
        flat_image = image.flatten()
        images.append(flat_image)
        labels.append(label)
    
    return np.array(images), np.array(labels)

X_train, y_train = dataset_to_numpy(train_dataset)
X_val, y_val = dataset_to_numpy(val_dataset)

In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report


# Initialize and train the model
clf = DecisionTreeClassifier(max_depth=10, random_state=42)
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_val)

# Accuracy
acc = accuracy_score(y_val, y_pred)
print("Validation Accuracy:", acc)

# Classification report
print(classification_report(y_val, y_pred))

Validation Accuracy: 0.37083641746854185
              precision    recall  f1-score   support

           0       0.48      0.46      0.47       116
           1       0.37      0.36      0.36       101
           2       0.21      0.50      0.29       115
           3       0.43      0.22      0.29       111
           4       0.30      0.39      0.34       129
           5       0.33      0.45      0.38       106
           6       0.26      0.36      0.30       104
           7       0.46      0.33      0.39       115
           8       0.21      0.26      0.23        97
           9       0.32      0.57      0.41       103
          10       0.32      0.26      0.29       121
          11       0.29      0.39      0.33        95
          12       0.60      0.62      0.61        93
          13       0.24      0.24      0.24       127
          14       0.31      0.38      0.34       108
          15       0.53      0.26      0.35       113
          16       0.58      0.71      0

This current model shows an accuracy of 37%. Lets see what we can do to improve it.

### 3.2 Random Forests and Extra Trees

Normal Decision Trees tend to overfit. Random Forests and Extra Trees try to prevent this by:
   - **Random Forests**: Build many trees using bootstrapped samples and random feature subsets.
- **Extra Trees**: Like random forests, but split thresholds are chosen randomly, not optimally.

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier

rf = RandomForestClassifier(
    n_estimators=200,         # number of trees
    max_depth=20,             # limit depth to avoid overfitting
    random_state=42,
    n_jobs=-1                 # use all CPU cores
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)

et = ExtraTreesClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

et.fit(X_train, y_train)
y_pred_et = et.predict(X_val)

from sklearn.metrics import accuracy_score
print("Random Forest Accuracy:", accuracy_score(y_val, y_pred_rf))
print("Extra Trees Accuracy:", accuracy_score(y_val, y_pred_et))

Random Forest Accuracy: 0.7900320750061682
Extra Trees Accuracy: 0.8211201579077226


Using these models we get an accuracy of 79% for Random Forests and 82% for extra Trees. Since Extra Trees got the better accuracy we will now try to further increase it's Accuracy.

### 3.3 Feature Extraction

Instead of changing the method in which we aquire our tree, we now want to try to reduce the dimension of our data by extracting only relevant features. 

The `extract_features(image)` function processes a single **grayscale image** and returns a **1D NumPy array** of features suitable for our Extra Trees. It extracts the following types of features:

#### 🔹 1. Intensity Histogram (32 bins)
Captures the global distribution of pixel intensities across the image.
- Input: flattened grayscale image.
- Output: 32 normalized histogram values.

#### 🔹 2. Tiled Average Intensities (8x8)
Captures local brightness patterns by dividing the image into 8x8 tiles and computing the average intensity in each tile.
- Output: 64 values (one per tile).

#### 🔹 3. Hu Moments (7 values)
Shape descriptors that are invariant to translation, scale, and rotation.
- Extracted from the largest external contour in the image.
- Output: 7 Hu moment values (or zeros if no contour is found).

#### 🔹 4. Gradient Magnitude Histogram (16 bins)
Captures edge strength information using Sobel operators in both x and y directions.
- Gradient magnitude is binned into a 16-bin histogram.
- Output: 16 normalized histogram values.

#### 🔹 5. Local Binary Pattern (LBP) Histogram
Describes local texture by comparing each pixel with its neighbors.
- Uses 8 neighbors and radius = 1 with `'uniform'` method.
- Output: 10-bin LBP histogram (normalized).

#### ✅ Returns
- A single 1D NumPy array of concatenated feature values.
- Feature vector combines intensity, texture, shape, and edge information.


In [13]:
import numpy as np
import cv2
from skimage.feature import local_binary_pattern

def extract_features(image):
    """
    Extracts features from a single grayscale image:
    - Intensity histogram (32 bins)
    - Tiled average intensities (8x8 tiles)
    - Hu moments
    Returns a flat 1D numpy array.
    """
    img = image.squeeze()  # shape: (H, W)
    features = []

    # Normalize image
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # Histogram (global intensity)
    hist = np.histogram(img.flatten(), bins=32, range=(0, 255))[0]
    hist = hist / (np.sum(hist) + 1e-6)  # normalize
    features.extend(hist)

    # Tiled average intensities (8x8 tiles)
    tile_h, tile_w = img.shape[0] // 8, img.shape[1] // 8
    for i in range(0, img.shape[0], tile_h):
        for j in range(0, img.shape[1], tile_w):
            tile = img[i:i+tile_h, j:j+tile_w]
            features.append(np.mean(tile))

    # Hu moments (invariant shape)
    contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        moments = cv2.moments(cnt)
        hu = cv2.HuMoments(moments).flatten()
    else:
        hu = np.zeros(7)
    features.extend(hu)

    # Gradient histogram
    sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobelx**2 + sobely**2)
    grad_hist = np.histogram(grad_mag, bins=16, range=(0, grad_mag.max() + 1e-6))[0]
    grad_hist = grad_hist / (np.sum(grad_hist) + 1e-6)
    features.extend(grad_hist)


    return np.array(features, dtype=np.float32)

In [14]:
def convert_dataset(dataset):
    X, y = [], []
    for item in dataset:
        img = item['image']
        features = extract_features(img)
        X.append(features)
        y.append(item['label'])
    return np.array(X), np.array(y)

Let's train our model using our new extracted feature dataset. Furthemore we will train another model again using the already trained Extra Trees using the SelectFromModel function to select only the most important features

In [15]:
# Extract features
X_train, y_train = convert_dataset(train_dataset)
X_val, y_val = convert_dataset(val_dataset)

In [19]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier


# Train
clf = RandomForestClassifier(
    n_estimators=250,
    max_depth=15,
    min_samples_split=3,
    max_features='sqrt',
    n_jobs=-1,
    random_state=55,
    class_weight='balanced'  # Handle class imbalance
)

clf.fit(X_train, y_train)


# Step 3: Select important features
selector = SelectFromModel(clf, threshold="mean", prefit=True)
X_train_reduced = selector.transform(X_train)
X_val_reduced = selector.transform(X_val)



# Step 4: Retrain model on reduced features
clf_reduced = RandomForestClassifier(
    n_estimators=225,
    max_depth=15,
    min_samples_split=3,
    max_features='sqrt',
    n_jobs=-1,
    random_state=56,
    class_weight='balanced'  # Handle class imbalance
)
clf_reduced.fit(X_train_reduced, y_train)

# Evaluate
y_pred = clf.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {acc*100:.2f}%")
print(classification_report(y_val, y_pred))


y_pred = clf_reduced.predict(X_val_reduced)
acc = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy (Reduced Features): {acc*100:.2f}%")
print(classification_report(y_val, y_pred))

Validation Accuracy: 83.54%
              precision    recall  f1-score   support

           0       0.89      0.84      0.87       100
           1       0.72      0.83      0.77        83
           2       0.83      0.80      0.82       108
           3       0.90      0.87      0.88       108
           4       0.66      0.70      0.68       113
           5       0.80      0.92      0.86       111
           6       0.68      0.73      0.70       105
           7       0.82      0.81      0.82       116
           8       0.73      0.65      0.69       104
           9       0.89      0.90      0.89       120
          10       0.77      0.83      0.80       120
          11       0.87      0.85      0.86       124
          12       0.91      0.99      0.95        94
          13       0.77      0.70      0.74       121
          14       0.92      0.90      0.91       127
          15       0.90      0.90      0.90       134
          16       0.81      0.88      0.84       109

Using this approach with extra trees we can achieve an accuracy of up to 90% (n_samples = 500 and max_size = 20).
Sadly such a model has a size of about 150 MB which superseeds the imposes 50 MB model size limit.

Reducing n_samples and max_size and using only random forests (less size) achieves an accuracy of 83 %

Because we do not want to introduce to much bias into our system, we conclude our training and save the model

## 4. Saving the model 

Because we trained our model using scikit, the Python library joblib is the best apporach for saving the parameters

In [20]:
import joblib
import os

model_path = "../models/decision_tree.pkl"

# Save the model
joblib.dump(clf_reduced, model_path,  compress=('xz', 9))

['../models/decision_tree.pkl']